In [1]:
import torch

# Step 1: Load the Data

In [2]:
words = open('names.txt', 'r').read().splitlines()
# ['emma', 'olivia', 'ava', ...] — 32,033 names

# Step 2: Count Bigrams (Character Pairs)

We treat each name as a sequence of characters with special start/end tokens:

"emma"  →  . e m m a .

**We count every adjacent pair:**

In [3]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']  # <S>=start, <E>=end
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [4]:
b

{('<S>', 'e'): 1531,
 ('e', 'm'): 769,
 ('m', 'm'): 168,
 ('m', 'a'): 2590,
 ('a', '<E>'): 6640,
 ('<S>', 'o'): 394,
 ('o', 'l'): 619,
 ('l', 'i'): 2480,
 ('i', 'v'): 269,
 ('v', 'i'): 911,
 ('i', 'a'): 2445,
 ('<S>', 'a'): 4410,
 ('a', 'v'): 834,
 ('v', 'a'): 642,
 ('<S>', 'i'): 591,
 ('i', 's'): 1316,
 ('s', 'a'): 1201,
 ('a', 'b'): 541,
 ('b', 'e'): 655,
 ('e', 'l'): 3248,
 ('l', 'l'): 1345,
 ('l', 'a'): 2623,
 ('<S>', 's'): 2055,
 ('s', 'o'): 531,
 ('o', 'p'): 95,
 ('p', 'h'): 204,
 ('h', 'i'): 729,
 ('<S>', 'c'): 1542,
 ('c', 'h'): 664,
 ('h', 'a'): 2244,
 ('a', 'r'): 3264,
 ('r', 'l'): 413,
 ('l', 'o'): 692,
 ('o', 't'): 118,
 ('t', 't'): 374,
 ('t', 'e'): 716,
 ('e', '<E>'): 3983,
 ('<S>', 'm'): 2538,
 ('m', 'i'): 1256,
 ('a', 'm'): 1634,
 ('m', 'e'): 818,
 ('<S>', 'h'): 874,
 ('r', 'p'): 14,
 ('p', 'e'): 197,
 ('e', 'r'): 1958,
 ('r', '<E>'): 1377,
 ('e', 'v'): 463,
 ('v', 'e'): 568,
 ('l', 'y'): 1588,
 ('y', 'n'): 1826,
 ('n', '<E>'): 6763,
 ('b', 'i'): 217,
 ('i', 'g'): 428,


# Step 3: Build a Tensor of Counts (N)


**Instead of a dict, we use a 27×27 matrix (26 letters + 1 for the . boundary token).**

N is a 27×27 table of counts. Each cell N[i, j] answers:

"How many times did character i appear immediately followed by character j in all the names?"


In [12]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}  # string → int
stoi['.'] = 0                                # '.' is index 0
itos = {i:s for s,i in stoi.items()}         # int → string

N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

# Step 4: Convert Counts to Probabilities


In [7]:
P = (N + 1).float()          # +1 smoothing (avoids log(0))
P = P / P.sum(1, keepdim=True)  # normalize each row to sum to 1

cexze.
momasurailezitynn.
konimittain.
llayn.
ka.


tensor(2.4544)

# Step 7: Replace Counting with a Neural Network


In [10]:
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)  # input characters (indices 0-26)
ys = torch.tensor(ys)  # target next characters
num = xs.nelement()    # 228,146 total bigrams

In [11]:
import torch.nn.functional as F
W = torch.randn((27, 27), requires_grad=True)

for k in range(100):
    # Forward pass
    xenc = F.one_hot(xs, num_classes=27).float()  # (num, 27)
    logits = xenc @ W                              # (num, 27)
    counts = logits.exp()                          # softmax numerator
    probs = counts / counts.sum(1, keepdims=True)  # softmax denominator

    # Loss: negative log-likelihood of the correct next character
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())
    
    # Backward pass
    W.grad = None
    loss.backward()

    # Update
    W.data += -50 * W.grad

3.7143874168395996
3.358769178390503
3.1477489471435547
3.0114948749542236
2.919956922531128
2.8516898155212402
2.799290180206299
2.758389472961426
2.725822687149048
2.699296712875366
2.6772170066833496
2.65850567817688
2.6424295902252197
2.6284756660461426
2.61626935005188
2.6055257320404053
2.5960206985473633
2.5875725746154785
2.5800297260284424
2.573265552520752
2.5671727657318115
2.5616605281829834
2.5566511154174805
2.552079677581787
2.5478899478912354
2.544034242630005
2.540472984313965
2.5371716022491455
2.5341012477874756
2.5312366485595703
2.5285561084747314
2.5260415077209473
2.523676633834839
2.5214476585388184
2.5193424224853516
2.5173499584198
2.5154616832733154
2.5136682987213135
2.511963367462158
2.5103399753570557
2.5087924003601074
2.507315158843994
2.505903959274292
2.504554271697998
2.503262519836426
2.5020244121551514
2.500838279724121
2.499699831008911
2.4986066818237305
2.4975571632385254
2.4965481758117676
2.4955780506134033
2.4946444034576416
2.493745803833008
